# nanoWeather v0: results

nanoWeather v0 trains a small vision transformer (about 5M parameters) to forecast the global atmosphere 6 hours ahead,
from ERA5 at 5.625° (a 32 × 64 grid), and runs it step by step to forecast 10 days ahead. One training run takes
minutes on a single GPU.

This notebook walks through the study: how each change to the original v0 improved the forecasts, what did not help,
and what the remaining errors look like.

**To reproduce the results** (from the repo root):

```bash
uv run scripts/prepare_data.py                                              # 29 variables
uv run scripts/prepare_data.py --config experiments/configs/data_6vars.yaml  # 6 variables
uv run scripts/run_experiments.py                                           # trains the missing models
uv run scripts/evaluate.py --experiments experiments/experiments.yaml --output_dir results/experiments
uv run scripts/diagnose.py --checkpoint checkpoints/v0/best
uv run scripts/diagnose.py --checkpoint checkpoints/vars29_both_rollout/best
uv run scripts/visualize.py --checkpoint checkpoints/vars29_both_rollout/best
```

In [ ]:
import os
from pathlib import Path


if Path.cwd().name == "notebooks":
    os.chdir("..")

In [ ]:
import glob
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from nanoweather.config import load_config
from nanoweather.data.datasets import get_store_path
from nanoweather.experiments import load_experiments
from nanoweather.visualization.forecast_figure import load_land_sea_mask


REGISTRY = "experiments/experiments.yaml"
RESULTS = "results/experiments"
DIAGNOSTICS = "results/diagnostics"
FIRST, FINAL = "v0", "vars29_both_rollout"
VARIABLES = ["z500", "t850", "t2m"]
UNITS = {"z500": "m²/s²", "t850": "K", "t2m": "K"}

In [ ]:
experiments = load_experiments(REGISTRY)
by_name = {experiment.name: experiment for experiment in experiments}


def load_results(name):
    path = os.path.join(RESULTS, f"{name}.json")
    if not os.path.exists(path):
        return None
    with open(path) as reader:
        return json.load(reader)


results = {name: load_results(name) for name in [*by_name, "persistence", "climatology"]}
missing = [name for name, result in results.items() if result is None]
print(f"{len(results) - len(missing)} results loaded" + (f"; missing: {', '.join(missing)}" if missing else ""))
lead_hours = np.array(next(result for result in results.values() if result)["lead_hours"])


def horizon_days(name, variable):
    """Lead time in days at which ACC falls below the threshold; inf if it stays above."""
    hours = results[name]["skill_horizon_hours"][variable]
    return np.inf if hours is None else hours / 24


def rmse_at(name, variable, days):
    return results[name]["rmse"][variable][list(lead_hours).index(days * 24)]


def trained(names):
    return [name for name in names if results.get(name) is not None]

## How forecasts are scored

Every model is scored the same way: forecasts started at 00 and 12 UTC on every day of 2020, compared with ERA5 up to
10 days ahead, on the 6 variables all models share. The three variables shown here are z500 (the height of the
500 hPa pressure level, which describes the large-scale flow), t850 (temperature around 1.5 km) and t2m (temperature
at 2 m).

- **RMSE**: typical size of the error, with each grid cell weighted by its area. Lower is better.
- **ACC** (anomaly correlation): how well the forecast's departures from the usual weather for that date match the real
  ones. 1 is perfect; 0 means no skill beyond the usual weather.
- **Skill horizon**: the lead time at which ACC falls below 0.6, the usual limit of a useful forecast. This is the main
  number to track. State-of-the-art models reach about 9-10 days for z500.

Two baselines: **persistence** (the weather stays as it is) and **climatology** (the average weather for that date and
hour, 1990-2019).

## 1. From v0 to the final model

Each step adds one change to the previous one.

In [ ]:
rows = []
for experiment in experiments:
    if experiment.group != "progression" or results[experiment.name] is None:
        continue
    row = {"model": experiment.name, "change": experiment.description}
    for variable in VARIABLES:
        row[f"{variable} horizon (days)"] = horizon_days(experiment.name, variable)
    row["z500 RMSE day 3"] = rmse_at(experiment.name, "z500", 3)
    row["z500 RMSE day 5"] = rmse_at(experiment.name, "z500", 5)
    rows.append(row)
progression = pd.DataFrame(rows).set_index("model")
progression.insert(2, "z500 horizon gain", progression["z500 horizon (days)"].diff())
formats = {column: "{:.1f}" for column in progression.columns if "horizon" in column}
formats["z500 horizon gain"] = "{:+.1f}"
formats.update({"z500 RMSE day 3": "{:.0f}", "z500 RMSE day 5": "{:.0f}"})
progression.style.format(formats, na_rep="")

In [ ]:
steps = list(progression.index)
x = np.arange(len(steps))
fig, ax = plt.subplots(figsize=(10, 4))
for offset, variable in zip((-0.25, 0, 0.25), VARIABLES):
    ax.bar(x + offset, progression[f"{variable} horizon (days)"], width=0.25, label=variable)
ax.set_xticks(x, steps, rotation=15)
ax.set_ylabel("skill horizon (days, ACC > 0.6)")
ax.set_title("Days of useful forecast at each step")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
shown = ["persistence", "climatology", *trained([e.name for e in experiments if e.group == "progression"])]
colors = dict(zip(shown, plt.rcParams["axes.prop_cycle"].by_key()["color"]))
fig, axes = plt.subplots(2, len(VARIABLES), figsize=(15, 8), constrained_layout=True)
for column, variable in enumerate(VARIABLES):
    for name in shown:
        days = lead_hours / 24
        axes[0, column].plot(days, results[name]["rmse"][variable], label=name, color=colors[name])
        acc = results[name].get("acc", {}).get(variable)
        if acc is not None and any(value is not None for value in acc):
            values = [np.nan if value is None else value for value in acc]
            axes[1, column].plot(days, values, label=name, color=colors[name])
    axes[0, column].set_title(f"{variable} RMSE ({UNITS[variable]})")
    axes[1, column].set_title(f"{variable} ACC")
    axes[1, column].axhline(0.6, color="k", linestyle="--", linewidth=0.8)
    for row in range(2):
        axes[row, column].set_xlabel("lead time (days)")
        axes[row, column].grid(alpha=0.3)
axes[0, 0].legend()
plt.show()

**What each step did**

- **29 variables** (winds, humidity and temperature at 5 levels instead of 2 fields): the model sees how the atmosphere
  is moving, so the large-scale flow (z500) improves, and for the first time the error grows more slowly. But t2m got
  worse: the loss is now shared among 29 variables, and the upper-air ones dominate it.
- **Loss on the change, normalized per variable**: each variable's predicted 6h change is scaled by its typical 6h
  change, so all variables count comparably in the loss. This is the single most important fix: it brings t2m back
  and improves everything else. GraphCast, Stormer and WeatherNext 2 all do this.
- **Pressure weights**: upper-air levels are weighted by pressure (each variable counting once in total) and t2m gets
  more weight than the other surface variables, as in GraphCast. This shifts priority to t2m, at a small cost in
  upper-air skill.
- **Rollout fine-tuning**: the model is trained on 2, then 4, then 8 consecutive steps, feeding its own predictions back
  as inputs. Trained only on one step from perfect inputs, it never learned to handle its own errors, which then grow
  quickly over a forecast. This adds close to a day of useful skill for a few minutes of training.

## 2. What did not help

Each ablation changes one thing compared with a reference model. Differences in skill horizon are in days, RMSE
differences in percent (negative is better). The two runs that differ only by their random seed show how large a
difference must be to mean something.

In [ ]:
rows = []
for experiment in experiments:
    reference = experiment.compare_to
    if experiment.group != "ablations" or results[experiment.name] is None or results.get(reference) is None:
        continue
    row = {"model": experiment.name, "change": experiment.description, "compared with": reference}
    for variable in VARIABLES:
        row[f"{variable} horizon (days)"] = horizon_days(experiment.name, variable) - horizon_days(reference, variable)
    for variable in ("z500", "t2m"):
        row[f"{variable} RMSE day 3 (%)"] = 100 * (
            rmse_at(experiment.name, variable, 3) / rmse_at(reference, variable, 3) - 1
        )
    rows.append(row)
ablations = pd.DataFrame(rows).set_index("model")
ablations.style.format(dict.fromkeys(ablations.columns[2:], "{:+.1f}")).background_gradient(
    cmap="RdBu", subset=[column for column in ablations.columns if "horizon" in column], vmin=-1, vmax=1
)

**Why these did not help**

- **Model size and training length**: smaller models are clearly worse, but larger ones and longer training give only a
  few percent with 6 variables. The limit was not the model: it lacked information and was trained on the wrong task.
- **Time features** (sunlight and time of year as inputs): no gain. The model already infers the time of day from the
  temperature field, where the warm side of the globe shows where the sun is.
- **Patches of 1 grid point**: no change. The excess small-scale noise seen in the spectra is not caused by the 2 × 2
  patches.
- **Two input time steps** (t - 6h and t): better for the first step, worse afterwards, see below.

In [ ]:
pairs = [("vars29_both_2steps", "vars29_both"), ("vars29_both_2steps_rollout", "vars29_both_rollout")]
pairs = [(a, b) for a, b in pairs if results.get(a) and results.get(b)]
if pairs:
    fig, axes = plt.subplots(1, len(VARIABLES), figsize=(15, 3.8), constrained_layout=True)
    for ax, variable in zip(axes, VARIABLES):
        for two_steps, one_step in pairs:
            ratio = np.array(results[two_steps]["rmse"][variable]) / np.array(results[one_step]["rmse"][variable])
            ax.plot(lead_hours / 24, 100 * (ratio - 1), label=f"{two_steps} vs {one_step}")
        ax.axhline(0, color="k", linewidth=0.8)
        ax.set_title(f"{variable}: RMSE difference, two vs one input steps (%)")
        ax.set_xlabel("lead time (days)")
        ax.grid(alpha=0.3)
    axes[0].legend(fontsize=8)
    plt.show()

Below zero, two input steps are better. At the first step, both inputs are real states and the extra one helps. From
the second or third step on, both inputs are the model's own predictions, and the advantage turns into a loss: the
model learned to rely on the difference between two real states, and extrapolates its own errors instead. Rollout
fine-tuning, which trains the model on its own predictions, removes most of the gap, but with winds already among the
inputs the previous state adds too little to come out ahead.

## 3. Where the errors are

Diagnostics over all forecasts of 2020, comparing the original v0 with the final model.

In [ ]:
def load_diagnostics(name):
    path = os.path.join(DIAGNOSTICS, name, "diagnostics.npz")
    return dict(np.load(path)) if os.path.exists(path) else None


diagnostics = {name: load_diagnostics(name) for name in (FIRST, FINAL)}
diagnostics = {name: value for name, value in diagnostics.items() if value is not None}
print("diagnostics for:", ", ".join(diagnostics) or "none (run scripts/diagnose.py)")

**Error by latitude.** Errors are small in the tropics and large in the mid-latitude storm tracks, where weather
changes fastest. They also rise toward both poles. Part of that is natural variability, but part comes from the
latitude-longitude grid: the rows near the poles are 64 points squeezed into a tiny area, they get almost no weight in
the loss, and points just across the pole are far apart in the grid. Models built on a mesh over the sphere
(GraphCast, WeatherNext 2) avoid this, which is the plan for v1.

In [ ]:
if diagnostics:
    fig, axes = plt.subplots(1, len(VARIABLES), figsize=(15, 3.8), constrained_layout=True)
    for name, values in diagnostics.items():
        names = list(values["variables"])
        last = len(values["lead_hours"]) - 1
        for ax, variable in zip(axes, VARIABLES):
            ax.plot(values["latitude"], values["rmse_by_latitude"][last, names.index(variable)], label=name)
            ax.set_title(f"{variable} RMSE by latitude, day {values['lead_hours'][last] / 24:g} ({UNITS[variable]})")
            ax.set_xlabel("latitude")
            ax.grid(alpha=0.3)
    axes[0].legend()
    plt.show()

**Spectra.** The power of the forecast at each scale (zonal wavenumber 1 is the size of the globe, 32 is two grid
points), compared with the truth. Two things to look for: forecast curves *below* the truth at small scales mean the
forecasts are smoother than reality, a known side effect of training with a squared error over several days (the
model hedges when it is unsure). Curves *above* the truth at the smallest scales mean grid-scale noise that builds up
during the forecast.

In [ ]:
if diagnostics:
    fig, axes = plt.subplots(1, len(VARIABLES), figsize=(15, 3.8), constrained_layout=True)
    for (name, values), style in zip(diagnostics.items(), ("--", "-")):
        names = list(values["variables"])
        last = len(values["lead_hours"]) - 1
        wavenumbers = np.arange(values["truth_spectrum"].shape[-1])[1:]
        for ax, variable in zip(axes, VARIABLES):
            index = names.index(variable)
            if style == "-":
                ax.loglog(wavenumbers, values["truth_spectrum"][last, index, 1:], "k", label="truth")
            ax.loglog(
                wavenumbers,
                values["forecast_spectrum"][last, index, 1:],
                style,
                label=f"{name}, day {values['lead_hours'][last] / 24:g}",
            )
            ax.set_title(f"{variable} zonal power spectrum")
            ax.set_xlabel("zonal wavenumber")
            ax.grid(alpha=0.3, which="both")
    axes[0].legend(fontsize=8)
    plt.show()

**Bias of the final model.** The average of forecast minus truth over all forecasts of 2020: where the model is
consistently too warm (red) or too cold (blue). Over high terrain (Antarctica, Greenland, Tibet), t850 is below the
ground and ERA5 only provides an extrapolated value, which is hard to predict.

In [ ]:
if FINAL in diagnostics:
    values = diagnostics[FINAL]
    names = list(values["variables"])
    data_config = load_config("configs/data.yaml")
    try:
        land_sea_mask = load_land_sea_mask(get_store_path(data_config["output_dir"], data_config["store_name"]))
    except Exception:
        land_sea_mask = None
    last = len(values["lead_hours"]) - 1
    fig, axes = plt.subplots(1, len(VARIABLES), figsize=(16, 3.4), constrained_layout=True)
    for ax, variable in zip(axes, VARIABLES):
        bias = values["bias"][last, names.index(variable)]
        limit = np.abs(bias).max()
        mesh = ax.pcolormesh(values["longitude"], values["latitude"], bias, cmap="RdBu_r", vmin=-limit, vmax=limit)
        if land_sea_mask is not None:
            ax.contour(
                values["longitude"], values["latitude"], land_sea_mask, levels=[0.5], colors="k", linewidths=0.5
            )
        ax.set_title(f"{variable} bias, day {values['lead_hours'][last] / 24:g}")
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(mesh, ax=ax, label=UNITS[variable])
    plt.show()

## 4. One forecast

The final model's forecast next to the truth (ERA5), and their difference, for one starting date.

In [ ]:
animations = sorted(glob.glob("results/forecast_*.gif"))
if animations:
    display(Image(filename=animations[0]))
else:
    print("No animation found: run scripts/visualize.py")

## Summary

In [ ]:
if results.get(FIRST) and results.get(FINAL):
    for variable in VARIABLES:
        before, after = horizon_days(FIRST, variable), horizon_days(FINAL, variable)
        print(f"{variable}: skill horizon {before:.1f} -> {after:.1f} days ({after - before:+.1f})")

**What worked**: giving the model a fuller description of the atmosphere, normalizing the loss per variable on the
predicted change, and training it on its own multi-step forecasts. **What did not**: larger models, longer training,
time inputs, smaller patches and a second input time step, each for an identifiable reason.

**Limits of v0**

- The resolution (5.625°, about 600 km per grid cell) sets a floor on what any model can do here.
- The poles stay the weakest region, a consequence of the latitude-longitude grid.
- The model gives a single forecast, trained with a squared error, so it gets smoother rather than sharper when unsure.
- Most comparisons are single runs; the seed comparison shows run-to-run differences are small.

**Next: v1** at 1.5°, with the mesh architecture of WeatherNext 2 (in transformers) and probabilistic training, with
this model as the baseline to beat.